# G17 AMFU-Net — loss-function search (R2)
U-Net + DAG, 200 epochs, cosine, seed 42; **test set pinned to the exact 266 scans of Exp-16–82** (patients excluded from training).
Two runs train in parallel (one per T4). Every model is scored with four post-processing variants
(`plain`, `lcc` = largest connected component, `tta` = horizontal-flip TTA, `tta+lcc`) and Boundary F1 @2 px.
Outputs: `runs/<name>/summary.csv`, `leaderboard.csv`, `overlays.png`.

In [ ]:
!pip -q install datasets safetensors huggingface_hub
import torch; print(torch.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
%%writefile g17core.py
# ============================================================
# Core code: data, model, losses, training, evaluation
# (architecture + split copied from training/training_marimo.py)
# ============================================================
import gc, json, math, os, random, time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms.functional as TF
from scipy.ndimage import binary_erosion, distance_transform_edt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG = 256
MODALITY_COL = {"raw": "raw_image", "clahe": "clahe_image", "srad": "srad_image", "enhanced": "image"}


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)


# ---------------- Data: the exact 266-scan test set used by Exp-16–82 ----------------
# StratifiedGroupKFold(shuffle=True) gives different folds on different scikit-learn versions
# (on Kaggle it yields 255 test scans, not 266). So the test set is pinned to the image list
# saved with Exp-16, every patient in it is excluded from train/val, and the rest is split as before.
def load_splits(data_repo="Ahorts/G17-preprocessed-dataset", seed=42,
                model_repo="Ahorts/G17-AMFU-Net-Paper",
                pin_branch="exp16-unet-dag-gpu-reflect-raw-cosine-only"):
    from datasets import concatenate_datasets, load_dataset
    from huggingface_hub import hf_hub_download
    from sklearn.model_selection import StratifiedGroupKFold
    hf = load_dataset(data_repo, revision="11b552497ffa661fa16c58c95bd7415b31073721")  # dataset as used for Exp-16–82
    allds = concatenate_datasets([hf["train"], hf["validation"], hf["test"]])
    ref = pd.read_csv(hf_hub_download(model_repo, "results/internal_test_per_image.csv", revision=pin_branch))
    want = set(zip(ref["dataset"].astype(str), ref["image_id"].astype(str)))
    keys = list(zip(np.asarray(allds["dataset"]).astype(str), np.asarray(allds["image_id"]).astype(str)))
    test_idx = np.array([i for i, k in enumerate(keys) if k in want])
    assert len(test_idx) == len(want) == 266, f"pinned test set mismatch: found {len(test_idx)} of {len(want)}"
    groups = np.asarray(allds["patient_id"]).astype(str)
    test_patients = set(groups[test_idx])
    rem = np.array([i for i in range(len(allds)) if groups[i] not in test_patients])
    labels = np.array([f"{d}_{c}" for d, c in zip(allds["dataset"], allds["class_label"])])
    tr, va = next(StratifiedGroupKFold(9, shuffle=True, random_state=seed + 1).split(rem, labels[rem], groups[rem]))
    split = {"train": allds.select(rem[tr].tolist()), "validation": allds.select(rem[va].tolist()),
             "test": allds.select(test_idx.tolist())}
    assert not (set(groups[rem[tr]]) | set(groups[rem[va]])) & test_patients, "patient leakage"
    print({k: len(v) for k, v in split.items()}, "| test pinned to", pin_branch, flush=True)
    return split


_TENSOR_CACHE = {}


def to_tensors(ds, modality, split_name):
    """Decode + resize once per (split, modality), keep uint8 on GPU like the original pipeline."""
    key = (split_name, modality)
    if key in _TENSOR_CACHE:
        return _TENSOR_CACHE[key]
    col = MODALITY_COL[modality]
    imgs, msks = [], []
    for r in ds:
        imgs.append(cv2.resize(np.asarray(r[col].convert("L")), (IMG, IMG), interpolation=cv2.INTER_AREA))
        msks.append(cv2.resize(np.asarray(r["mask"].convert("L")), (IMG, IMG), interpolation=cv2.INTER_NEAREST))
    out = {"img": torch.from_numpy(np.stack(imgs))[:, None].to(DEVICE),
           "msk": torch.from_numpy(np.stack(msks))[:, None].to(DEVICE),
           "dataset": list(ds["dataset"]), "image_id": list(ds["image_id"])}
    _TENSOR_CACHE[key] = out
    return out


def norm(img_u8):
    return (img_u8.float() / 255.0 - 0.5) / 0.5


def augment(img, msk):
    """Same augmentation as training: hflip, rot ±15, scale 0.9–1.1, translate ±10%,
    96-px reflection pad for the image, zero fill for the mask."""
    if random.random() < 0.5:
        img, msk = TF.hflip(img), TF.hflip(msk)
    angle, scale = random.uniform(-15, 15), random.uniform(0.9, 1.1)
    tx, ty = random.uniform(-0.1, 0.1) * IMG, random.uniform(-0.1, 0.1) * IMG
    p = 96
    imgp = F.pad(img[None].float(), (p, p, p, p), mode="reflect")[0]
    imgp = TF.affine(imgp, angle=angle, translate=[tx, ty], scale=scale, shear=0.0,
                     interpolation=TF.InterpolationMode.BILINEAR)
    img = imgp[:, p:p + IMG, p:p + IMG]
    msk = TF.affine(msk, angle=angle, translate=[tx, ty], scale=scale, shear=0.0,
                    interpolation=TF.InterpolationMode.NEAREST)
    return (img / 255.0 - 0.5) / 0.5, (msk > 127).float()


# ---------------- Model ----------------
class ConvBNReLU(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.block = nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

    def forward(self, x):
        return self.block(x)


class DAG(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.branch_average = nn.Conv2d(c, c, 3, padding=1)
        self.branch_maximum = nn.Conv2d(c, c, 3, padding=1)
        self.output = nn.Conv2d(c, c, 3, padding=1)

    def forward(self, x):
        g1 = F.adaptive_avg_pool2d(torch.sigmoid(self.branch_average(x)), 1)
        g2 = F.adaptive_max_pool2d(torch.sigmoid(self.branch_maximum(x)), 1)
        return self.output(x * g1 + x * g2)


class MFEF(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 1); self.conv3 = nn.Conv2d(c, c, 3, padding=1)
        self.conv5 = nn.Conv2d(c, c, 5, padding=2); self.gap_projection = nn.Conv2d(c, c, 1)
        self.compress = nn.Conv2d(c * 4, c, 1); self.dilated = nn.Conv2d(c, c, 3, padding=2, dilation=2)

    def forward(self, x):
        g = F.interpolate(self.gap_projection(F.adaptive_avg_pool2d(x, 1)), size=x.shape[-2:],
                          mode="bilinear", align_corners=False)
        w1 = torch.softmax(F.relu(self.compress(torch.cat([self.conv1(x), self.conv3(x), self.conv5(x), g], 1))), 1)
        w2 = torch.softmax(F.relu(self.dilated(x)), 1)
        return x * (0.5 * w1 + 0.5 * w2)


class AMFUNet(nn.Module):
    def __init__(self, base_channels=32, use_dag=True, use_mfef=False, **_):
        super().__init__()
        b = base_channels; self.use_dag, self.use_mfef = use_dag, use_mfef
        self.enc1, self.enc2 = ConvBNReLU(1, b), ConvBNReLU(b, b * 2)
        self.enc3, self.enc4 = ConvBNReLU(b * 2, b * 4), ConvBNReLU(b * 4, b * 8)
        self.bottleneck = ConvBNReLU(b * 8, b * 16); self.pool = nn.MaxPool2d(2)
        if use_mfef:
            self.mfef = MFEF(b * 16)
        if use_dag:
            self.dag1, self.dag2, self.dag3, self.dag4 = DAG(b), DAG(b * 2), DAG(b * 4), DAG(b * 8)
        self.up4, self.dec4 = nn.ConvTranspose2d(b * 16, b * 8, 2, 2), ConvBNReLU(b * 16, b * 8)
        self.up3, self.dec3 = nn.ConvTranspose2d(b * 8, b * 4, 2, 2), ConvBNReLU(b * 8, b * 4)
        self.up2, self.dec2 = nn.ConvTranspose2d(b * 4, b * 2, 2, 2), ConvBNReLU(b * 4, b * 2)
        self.up1, self.dec1 = nn.ConvTranspose2d(b * 2, b, 2, 2), ConvBNReLU(b * 2, b)
        self.head = nn.Conv2d(b, 1, 1)

    def forward(self, x):
        s1 = self.enc1(x); s2 = self.enc2(self.pool(s1)); s3 = self.enc3(self.pool(s2)); s4 = self.enc4(self.pool(s3))
        x = self.bottleneck(self.pool(s4))
        if self.use_mfef:
            x = self.mfef(x)
        for up, dec, s, k in ((self.up4, self.dec4, s4, 4), (self.up3, self.dec3, s3, 3),
                              (self.up2, self.dec2, s2, 2), (self.up1, self.dec1, s1, 1)):
            x = dec(torch.cat([up(x), getattr(self, f"dag{k}")(s) if self.use_dag else s], 1))
        return self.head(x)


# ---------------- Losses ----------------
def extract_boundary(t):
    """Differentiable morphological gradient (dilation − erosion), as in training_marimo.py."""
    return F.max_pool2d(t, 3, 1, 1) - (-F.max_pool2d(-t, 3, 1, 1))


def boundary_weight_at(epoch, lam, delay=0, ramp=0):
    """λ schedule: 0 until `delay` epochs, then linear ramp to λ over `ramp` epochs (Ahorts' wu10-20 runs)."""
    if lam <= 0:
        return 0.0
    if epoch < delay:
        return 0.0
    if ramp <= 0:
        return lam
    return lam * min(1.0, (epoch - delay) / ramp)


def compute_loss(logits, labels, cfg, bnd_w):
    logits, labels = logits.float(), labels.float()
    p = torch.sigmoid(logits)
    dims = (1, 2, 3)
    loss = F.binary_cross_entropy_with_logits(logits, labels)
    if cfg["base"] == "dice":                      # BCE + soft Dice (Exp-16 family)
        inter = (p * labels).sum(dims)
        dice = (2 * inter + 1.0) / (p.sum(dims) + labels.sum(dims) + 1.0)
        loss = loss + (1.0 - dice.mean())
    elif cfg["base"] == "ft":                      # BCE + Focal Tversky (Exp-32–47 family)
        a, b, g = cfg["alpha"], cfg["beta"], cfg["gamma"]
        tp = (p * labels).sum(dims); fp = (p * (1 - labels)).sum(dims); fn = ((1 - p) * labels).sum(dims)
        ti = (tp + 1.0) / (tp + a * fp + b * fn + 1.0)
        loss = loss + torch.pow(torch.clamp(1 - ti, 1e-7, 1.0), 1.0 / g).mean()
    if bnd_w > 0 and cfg.get("bnd_type", "morph") == "morph":   # Ahorts-style: soft Dice between boundary maps
        pb, lb = extract_boundary(p), extract_boundary(labels)
        bd = (2 * (pb * lb).sum(dims) + 1.0) / (pb.sum(dims) + lb.sum(dims) + 1.0)
        loss = loss + bnd_w * (1.0 - bd.mean())
    elif bnd_w > 0 and cfg["bnd_type"] == "sdf":                 # Kervadec et al. 2019 distance-map boundary loss
        loss = loss + bnd_w * (p * signed_distance(labels)).mean()
    return loss


def signed_distance(labels):
    """Signed distance to the GT contour, normalised by image size: <0 inside, >0 outside."""
    out = np.zeros(labels.shape, np.float32)
    for i, m in enumerate(labels[:, 0].detach().cpu().numpy() > 0.5):
        if m.any() and (~m).any():
            out[i, 0] = (distance_transform_edt(~m) - (distance_transform_edt(m) - 1) * m) / m.shape[0]
    return torch.from_numpy(out).to(labels.device)


# ---------------- Metrics (identical definitions to training) ----------------
def surface_distances(pred, tgt):
    pred, tgt = pred.astype(bool), tgt.astype(bool)
    if not pred.any() and not tgt.any():
        return 0.0, 0.0
    if not pred.any() or not tgt.any():
        d = float(math.hypot(*tgt.shape)); return d, d
    pb = pred ^ binary_erosion(pred, border_value=0); tb = tgt ^ binary_erosion(tgt, border_value=0)
    pb = pb if pb.any() else pred; tb = tb if tb.any() else tgt
    d = np.concatenate([distance_transform_edt(~tb)[pb], distance_transform_edt(~pb)[tb]])
    return float(np.percentile(d, 95)), float(d.mean())


def boundary_f1(pred, tgt, tol=2):
    """Boundary F1 at `tol` px (as in Ahorts' qualitative audit): contour precision/recall within tolerance."""
    pred, tgt = pred.astype(bool), tgt.astype(bool)
    if not pred.any() and not tgt.any():
        return 1.0
    if not pred.any() or not tgt.any():
        return 0.0
    pb = pred ^ binary_erosion(pred, border_value=0); tb = tgt ^ binary_erosion(tgt, border_value=0)
    pb = pb if pb.any() else pred; tb = tb if tb.any() else tgt
    prec = (distance_transform_edt(~tb)[pb] <= tol).mean()
    rec = (distance_transform_edt(~pb)[tb] <= tol).mean()
    return float(2 * prec * rec / (prec + rec + 1e-12))


def confident_cc(mask, prob, keep_prob=0.9, keep_frac=0.2):
    """Smarter filter: keep every component that is either confident (max prob >= keep_prob) or
    at least `keep_frac` of the largest component; drops small low-confidence islands only."""
    from scipy.ndimage import label
    lab, n = label(mask)
    if n <= 1:
        return mask
    sizes = np.bincount(lab.ravel()); sizes[0] = 0
    big = sizes.max(); keep = np.zeros(n + 1, bool)
    for k in range(1, n + 1):
        keep[k] = sizes[k] >= keep_frac * big or prob[lab == k].max() >= keep_prob
    return keep[lab]


def largest_cc(mask):
    """Keep only the largest connected component (removes distant false-positive islands)."""
    from scipy.ndimage import label
    lab, n = label(mask)
    if n <= 1:
        return mask
    sizes = np.bincount(lab.ravel()); sizes[0] = 0
    return lab == sizes.argmax()


@torch.no_grad()
def predict(model, data, bs=32, tta=False):
    model.eval(); out = []
    for i in range(0, len(data["img"]), bs):
        x = norm(data["img"][i:i + bs])
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            p = torch.sigmoid(model(x).float())
            if tta:
                p = 0.5 * (p + torch.flip(torch.sigmoid(model(torch.flip(x, [-1])).float()), [-1]))
        out.append(p.cpu())
    return torch.cat(out)[:, 0].numpy()


def fast_dice(probs, data, thr=0.5):
    pred = probs >= thr; gt = data["msk"][:, 0].cpu().numpy() > 127
    tp = (pred & gt).sum((1, 2)); fp = (pred & ~gt).sum((1, 2)); fn = (~pred & gt).sum((1, 2))
    return float(np.mean((2 * tp + 1e-7) / (2 * tp + fp + fn + 1e-7)))


def full_metrics(probs, data, thr=0.5, lcc=False):
    gt_all = data["msk"][:, 0].cpu().numpy() > 127; rows = []
    for i, (pr, gt) in enumerate(zip(probs >= thr, gt_all)):
        if lcc == "conf":
            pr = confident_cc(pr, probs[i])
        elif lcc:
            pr = largest_cc(pr)
        tp = (pr & gt).sum(); fp = (pr & ~gt).sum(); fn = (~pr & gt).sum()
        hd95, asd = surface_distances(pr, gt)
        rows.append({"image_id": data["image_id"][i], "dataset": data["dataset"][i],
                     "dice": (2 * tp + 1e-7) / (2 * tp + fp + fn + 1e-7),
                     "jaccard": (tp + 1e-7) / (tp + fp + fn + 1e-7),
                     "recall": (tp + 1e-7) / (tp + fn + 1e-7),
                     "precision": (tp + 1e-7) / (tp + fp + 1e-7),
                     "hd95": hd95, "asd": asd, "bf1": boundary_f1(pr, gt)})
    return pd.DataFrame(rows)


VARIANTS = {"plain": (False, False), "lcc": (False, True), "tta": (True, False), "tta+lcc": (True, True),
            "tta+conf": (True, "conf")}


def evaluate_all(model, data):
    """All four post-processing variants; returns {variant: per-image df} and the TTA probability maps."""
    out, probs = {}, {False: predict(model, data), True: predict(model, data, tta=True)}
    for v, (tta, lcc) in VARIANTS.items():
        out[v] = full_metrics(probs[tta], data, lcc=lcc)
    return out, probs


def summarise(df, name):
    rows = []
    for grp, d in [("Overall", df)] + list(df.groupby("dataset")):
        rows.append({"run": name, "cohort": grp, "n": len(d), "dice": d.dice.mean() * 100,
                     "iou": d.jaccard.mean() * 100, "hd95": d.hd95.mean(), "asd": d.asd.mean(),
                     "bf1": d.bf1.mean() * 100 if "bf1" in d else np.nan,
                     "fail_lt50": (d.dice < 0.5).mean() * 100, "dice_sd": d.dice.std() * 100})
    return pd.DataFrame(rows)


# ---------------- Training ----------------
def train_run(cfg, split, out_root, epochs, batch=4, accum=3, lr=1e-3, wd=1e-5, min_lr=1e-5,
              eval_every=250, use_amp=True):
    name = cfg["name"]; out = Path(out_root) / name; out.mkdir(parents=True, exist_ok=True)
    if (out / "test_per_image.csv").exists():
        print(f"[skip] {name} already finished", flush=True); return pd.read_csv(out / "test_per_image.csv")
    seed_everything(cfg.get("seed", 42))
    tr = to_tensors(split["train"], cfg["modality"], "train")
    va = to_tensors(split["validation"], cfg["modality"], "validation")
    te = to_tensors(split["test"], cfg["modality"], "test")
    model = AMFUNet(use_dag=cfg.get("dag", True), use_mfef=cfg.get("mfef", False)).to(DEVICE)
    nparams = sum(p.numel() for p in model.parameters())
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    n = len(tr["img"]); micro_per_epoch = math.ceil(n / batch)
    steps_per_epoch = math.ceil(micro_per_epoch / accum); total = steps_per_epoch * epochs
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total, eta_min=min_lr)
    scaler = torch.amp.GradScaler(enabled=use_amp and DEVICE.type == "cuda")
    best, best_step, step, log = -1.0, 0, 0, []
    print(f"\n=== {name} | {cfg['modality']} | base={cfg['base']} λ={cfg.get('bnd', 0)} | "
          f"params={nparams:,} | {epochs} ep × {steps_per_epoch} steps = {total} steps ===", flush=True)
    t0 = time.time()
    for ep in range(epochs):
        model.train(); perm = torch.randperm(n).tolist()
        bw = boundary_weight_at(ep, cfg.get("bnd", 0.0), cfg.get("bnd_delay", 0), cfg.get("bnd_ramp", 0))
        opt.zero_grad(set_to_none=True)
        for mi in range(micro_per_epoch):
            ids = perm[mi * batch:(mi + 1) * batch]
            pairs = [augment(tr["img"][j], tr["msk"][j]) for j in ids]
            x = torch.stack([a for a, _ in pairs]); y = torch.stack([b for _, b in pairs])
            with torch.autocast(DEVICE.type, enabled=use_amp and DEVICE.type == "cuda"):
                logits = model(x)
            loss = compute_loss(logits, y, cfg, bw) / accum
            scaler.scale(loss).backward()
            if (mi + 1) % accum == 0 or mi + 1 == micro_per_epoch:
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step(); step += 1
                if step % eval_every == 0 or step == total:
                    vd = fast_dice(predict(model, va), va); model.train()
                    log.append({"step": step, "epoch": ep + 1, "val_dice": vd, "lr": sched.get_last_lr()[0], "bnd_w": bw})
                    if vd > best:
                        best, best_step = vd, step
                        torch.save(model.state_dict(), out / "best.pt")
        if ep == 0 or (ep + 1) % 10 == 0:
            el = time.time() - t0
            print(f"  [{name}] ep {ep+1:>3}/{epochs}  val_dice(best)={best:.4f}@{best_step}  "
                  f"{el/60:.1f} min elapsed, ETA {el/(ep+1)*(epochs-ep-1)/60:.0f} min", flush=True)
    model.load_state_dict(torch.load(out / "best.pt", map_location=DEVICE))
    finish_eval(model, name, out, te, va)
    pd.DataFrame(log).to_csv(out / "val_log.csv", index=False)
    json.dump({**cfg, "params": nparams, "best_val_dice": best, "best_step": best_step, "epochs": epochs,
               "minutes": (time.time() - t0) / 60}, open(out / "config.json", "w"), indent=2)
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return pd.read_csv(out / "test_per_image.csv")


def finish_eval(model, name, out, te, va=None):
    """Evaluate test (and val) under every post-processing variant, save CSVs + TTA prob maps, print summary."""
    res, probs = evaluate_all(model, te)
    rows = []
    for v, df in res.items():
        df.to_csv(out / f"test_{v}.csv", index=False)
        rows.append(summarise(df, name).assign(variant=v, split="test"))
    res["plain"].to_csv(out / "test_per_image.csv", index=False)
    np.savez_compressed(out / "test_probs.npz", tta=(probs[True] * 255).astype(np.uint8),
                        plain=(probs[False] * 255).astype(np.uint8))
    if va is not None:
        vres, vprobs = evaluate_all(model, va)
        np.savez_compressed(out / "val_probs.npz", tta=(vprobs[True] * 255).astype(np.uint8),
                            plain=(vprobs[False] * 255).astype(np.uint8))
        for v, df in vres.items():
            rows.append(summarise(df, name).assign(variant=v, split="val"))
    summ = pd.concat(rows); summ.to_csv(out / "summary.csv", index=False)
    o = summ[(summ.cohort == "Overall")][["split", "variant", "dice", "iou", "hd95", "asd", "bf1", "fail_lt50"]]
    print(f"\n### RESULT {name}\n" + o.round(2).to_string(index=False), flush=True)
    t = summ[(summ.split == "test") & (summ.variant == "plain")][["cohort", "dice", "hd95", "asd", "bf1"]]
    print(t.round(2).to_string(index=False), flush=True)
    return summ


# ---------------- Re-evaluate existing Hugging Face checkpoints on the same split ----------------
def eval_hf_checkpoint(name, branch, split, out_root, repo="Ahorts/G17-AMFU-Net-Paper"):
    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    out = Path(out_root) / name; out.mkdir(parents=True, exist_ok=True)
    if (out / "summary.csv").exists():
        return pd.read_csv(out / "summary.csv")
    c = json.load(open(hf_hub_download(repo, "config.json", revision=branch)))
    model = AMFUNet(use_dag=c.get("use_dag", True), use_mfef=c.get("use_mfef", False)).to(DEVICE)
    model.load_state_dict(load_file(hf_hub_download(repo, "model.safetensors", revision=branch)), strict=True)
    modality = ("enhanced" if "enhanced" in branch else "clahe" if "clahe" in branch
                else "srad" if "srad" in branch else "raw")
    te = to_tensors(split["test"], modality, "test")
    return finish_eval(model, name, out, te)   # no val: old checkpoints were selected on a different val split


# ---------------- Ensembles: average TTA probability maps of several runs ----------------
def ensemble_eval(name, members, out_root, split, modality="raw"):
    out = Path(out_root) / name; out.mkdir(parents=True, exist_ok=True)
    rows = []
    for sp, key in (("test", "test"), ("val", "validation")):
        files = [Path(out_root) / m / f"{sp}_probs.npz" for m in members]
        if not all(f.exists() for f in files):
            continue
        p = np.mean([np.load(f)["tta"].astype(np.float32) / 255.0 for f in files], axis=0)
        data = to_tensors(split[key], modality, key)
        for v, lcc in (("tta", False), ("tta+conf", "conf"), ("tta+lcc", True)):
            df = full_metrics(p, data, lcc=lcc)
            if sp == "test":
                df.to_csv(out / f"test_{v}.csv", index=False)
            rows.append(summarise(df, name).assign(variant=v, split=sp))
        if sp == "test":
            np.savez_compressed(out / "test_probs.npz", tta=(p * 255).astype(np.uint8), plain=(p * 255).astype(np.uint8))
            full_metrics(p, data).to_csv(out / "test_plain.csv", index=False)
    summ = pd.concat(rows); summ.to_csv(out / "summary.csv", index=False)
    o = summ[summ.cohort == "Overall"][["split", "variant", "dice", "iou", "hd95", "asd", "bf1", "fail_lt50"]]
    print(f"\n### RESULT {name} = mean({', '.join(members)})\n" + o.round(2).to_string(index=False), flush=True)
    return summ


# ---------------- Parallel workers: one process per GPU ----------------
def worker(gpu, run_names, plan_path, out_root, epochs):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu)
    plan = {c["name"]: c for c in json.load(open(plan_path))}
    split = load_splits()
    for n in run_names:
        try:
            train_run(plan[n], split, out_root, epochs=epochs)
        except Exception as e:  # keep the other runs going
            import traceback; traceback.print_exc(); print(f"[FAILED] {n}: {e}", flush=True)


# ---------------- Ahorts-style contour overlays ----------------
def overlay_figure(out_root, run_names, split, case_ids, path, modality_of):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    te_raw = to_tensors(split["test"], "raw", "test")
    idx = [te_raw["image_id"].index(c) for c in case_ids]
    fig, axes = plt.subplots(len(idx), len(run_names), figsize=(2.6 * len(run_names), 2.6 * len(idx)), squeeze=False)
    gt_all = te_raw["msk"][:, 0].cpu().numpy() > 127
    per = {r: pd.read_csv(Path(out_root) / r / "test_plain.csv").set_index("image_id") for r in run_names}
    for c, r in enumerate(run_names):
        probs = np.load(Path(out_root) / r / "test_probs.npz")["plain"]
        for k, i in enumerate(idx):
            ax = axes[k, c]; ax.imshow(te_raw["img"][i, 0].cpu().numpy(), cmap="gray"); ax.axis("off")
            ax.contour(gt_all[i], levels=[0.5], colors="lime", linewidths=1.2)
            ax.contour(probs[i] >= 128, levels=[0.5], colors="magenta", linewidths=1.0)
            m = per[r].loc[te_raw["image_id"][i]]
            ax.set_title(f"{r[:18]}\nD {m.dice*100:.0f}  HD {m.hd95:.0f}  BF1 {m.bf1*100:.0f}", fontsize=7)
    fig.suptitle("green = ground truth, magenta = prediction", fontsize=9)
    plt.tight_layout(); plt.savefig(path, dpi=75); plt.close(fig)


if __name__ == "__main__":
    import sys
    worker(int(sys.argv[1]), sys.argv[2].split(","), sys.argv[3], sys.argv[4], int(sys.argv[5]))


In [ ]:
# Round 2 plan (revised 06:10) — U-Net + DAG, raw scans.
# R1 (val TTA / test TTA): ft55 82.06/85.90 | dice+morph0.5 82.09/85.81 | dice+morph0.25 82.08/85.57
#   repro 81.52/85.38 | ft a0.6b0.4 81.54/85.38 | SDF 81.42/85.34 | ft55+morph0.25 warm-up 81.75/85.07 (combo did NOT help)
# The top three are within noise (~0.5%) of each other → R2 = seed replicates (mean ± sd over 3 seeds)
# + ensembles fixed in advance, plus two cheap probes.
EPOCHS = 200
OUT = "/kaggle/working/runs"
REFERENCE = {}
FT55 = dict(base="ft", alpha=0.5, beta=0.5, gamma=1.33)
PLAN = [   # final R1 val TTA: ft a0.4b0.6 82.57 | dice+morph0.5 82.09 | dice+morph0.25 82.08 | ft55 82.06
    dict(name="R2_ft4060_s1",         modality="raw", base="ft", alpha=0.4, beta=0.6, gamma=1.33, seed=1),
    dict(name="R2_ft55_s1",           modality="raw", **FT55, seed=1),
    dict(name="R2_ft4060_s2",         modality="raw", base="ft", alpha=0.4, beta=0.6, gamma=1.33, seed=2),
    dict(name="R2_ft55_s2",           modality="raw", **FT55, seed=2),
    dict(name="R2_dice_morph05_s1",   modality="raw", base="dice", bnd=0.5, seed=1),
    dict(name="R2_ft4060_morph05",    modality="raw", base="ft", alpha=0.4, beta=0.6, gamma=1.33, bnd=0.5),  # probe: best FT + boundary, no warm-up
    dict(name="R2_dice_morph05_s2",   modality="raw", base="dice", bnd=0.5, seed=2),
    dict(name="R2_ft55_morph05",      modality="raw", **FT55, bnd=0.5),                                       # probe
]
ENSEMBLES = {   # fixed in advance; members chosen by R1 validation
    "ENS_ft4060_x3":   ["R1_ft_a04b06", "R2_ft4060_s1", "R2_ft4060_s2"],
    "ENS_ft55_x3":     ["R1_ft_a05b05", "R2_ft55_s1", "R2_ft55_s2"],
    "ENS_morph05_x3":  ["R1_dice_morph05", "R2_dice_morph05_s1", "R2_dice_morph05_s2"],
    "ENS_R1_top4":     ["R1_ft_a04b06", "R1_dice_morph05", "R1_dice_morph025", "R1_ft_a05b05"],
    "ENS_all9":        ["R1_ft_a04b06", "R2_ft4060_s1", "R2_ft4060_s2", "R1_ft_a05b05", "R2_ft55_s1", "R2_ft55_s2",
                        "R1_dice_morph05", "R2_dice_morph05_s1", "R2_dice_morph05_s2"],
}
print(len(PLAN), "runs ×", EPOCHS, "epochs")


In [ ]:
# Bring in previous rounds' runs (attach the previous notebook's output as Input) — needed for ensembles/leaderboard
import glob, shutil, os
os.makedirs(OUT, exist_ok=True)
for d in sorted(glob.glob("/kaggle/input/**/runs/*", recursive=True)):
    if os.path.isdir(d) and not os.path.exists(os.path.join(OUT, os.path.basename(d))):
        shutil.copytree(d, os.path.join(OUT, os.path.basename(d)))
print("previous runs:", sorted(os.listdir(OUT)))

In [ ]:
# Reference checkpoints first (fast) — scored on the identical test set with all variants
import g17core as g, pandas as pd, json, subprocess, os, time, sys
split = g.load_splits()
for n, b in REFERENCE.items():
    g.eval_hf_checkpoint(n, b, split, OUT)
import gc, torch; g._TENSOR_CACHE.clear(); gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Train: one worker process per GPU, runs alternate between GPUs
json.dump(PLAN, open("plan.json", "w"))
names = [c["name"] for c in PLAN]
ngpu = max(1, torch.cuda.device_count())
queues = [names[i::ngpu] for i in range(ngpu)]
procs = []
for gpu, q in enumerate(queues):
    if not q: continue
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
    log = open(f"worker{gpu}.log", "w")
    procs.append((subprocess.Popen([sys.executable, "g17core.py", str(gpu), ",".join(q), "plan.json", OUT, str(EPOCHS)],
                                   stdout=log, stderr=subprocess.STDOUT, env=env), f"worker{gpu}.log"))
    print(f"GPU{gpu}: {q}")
seen = {p: 0 for _, p in procs}
def pump():
    for _, p in procs:
        lines = open(p).read().splitlines()
        for ln in lines[seen[p]:]:
            if any(k in ln for k in ("===", "ep   1/", "0/200", "RESULT", "split", "test ", " val ", "Overall", "BUS", "FAILED", "Error", "Traceback")):
                print(ln, flush=True)
        seen[p] = len(lines)
while any(pr.poll() is None for pr, _ in procs):
    pump(); time.sleep(60)
pump(); print("all workers finished")

In [ ]:
# Ensembles fixed in advance (average of TTA probability maps)
for en, members in globals().get("ENSEMBLES", {}).items():
    if all(os.path.exists(os.path.join(OUT, m, "test_probs.npz")) for m in members):
        g.ensemble_eval(en, members, OUT, split)
    else:
        print("skip", en, "- missing members")

In [ ]:
# Leaderboard (test, overall) — every model × post-processing variant
rows = []
for d in sorted(os.listdir(OUT)):
    f = os.path.join(OUT, d, "summary.csv")
    if os.path.exists(f): rows.append(pd.read_csv(f))
board = pd.concat(rows); board.to_csv("leaderboard.csv", index=False)
t = board[(board.split == "test") & (board.cohort == "Overall")]
pd.set_option("display.width", 250)
print("=== LEADERBOARD test overall (sorted by dice) ===")
print(t.sort_values("dice", ascending=False)[["run", "variant", "dice", "iou", "hd95", "asd", "bf1", "fail_lt50"]].round(2).to_string(index=False))
print("=== per cohort (plain) ===")
print(board[(board.split == "test") & (board.variant == "plain")].pivot_table(index="run", columns="cohort", values="dice").round(2).to_string())
v = board[(board.split == "val") & (board.cohort == "Overall")]
if len(v):
    best = v.sort_values("dice", ascending=False).groupby("run").head(1)[["run", "variant", "dice"]]
    print("=== variant chosen on VALIDATION per new run ===\n" + best.round(2).to_string(index=False))

In [ ]:
# Seed statistics: mean ± sd across seeds of each configuration (test & val, TTA)
import re
tt = board[(board.cohort == "Overall") & (board.variant == "tta")].copy()
tt["config"] = tt.run.str.replace(r"^R[12]_", "", regex=True).str.replace(r"_s\d$", "", regex=True).replace({"ft_a05b05": "ft55", "ft_a04b06": "ft4060"})
agg = tt.groupby(["config", "split"]).agg(n=("dice", "size"), dice_mean=("dice", "mean"), dice_sd=("dice", "std"),
                                          hd95=("hd95", "mean"), asd=("asd", "mean"), bf1=("bf1", "mean")).round(2)
print("=== SEED STATS (TTA) ===\n" + agg[agg.n > 1].to_string())

In [ ]:
# Ahorts-style contour overlays on the hardest test cases (worst for Exp-16) + two typical ones
ref = pd.read_csv(os.path.join(OUT, "Exp16_ref", "test_plain.csv")).sort_values("dice")
cases = list(ref.image_id[:6]) + list(ref.image_id[len(ref)//2:len(ref)//2 + 2])
runs = [r for r in ["Exp16_ref", "R1_ft_a05b05", "R1_dice_morph05"] + names + list(globals().get("ENSEMBLES", {})) if os.path.exists(os.path.join(OUT, r, "test_probs.npz"))]
g.overlay_figure(OUT, runs, split, cases, "overlays.png", None)
from IPython.display import Image as I; display(I("overlays.png"))